In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils import (
    build_binary_relevance_maps,
    build_eval_items,
    build_global_corpus,
    evaluate_sparqlce_checkpoint,
    load_triplets,
)

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = os.path.join(PROJECT_ROOT, "dataset", "hard_triplets_tfidf_wikidata.json")
CHECKPOINT_PATH = os.path.join(PROJECT_ROOT, "../checkpoints", "wikidata_model_semantic-20.pt")
TOPKS = [3, 5, 10, 20]
BATCH = 32

raw_data = load_triplets(DATA_PATH)
global_corpus, seen_q = build_global_corpus(raw_data)
eval_items = build_eval_items(raw_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Loaded {len(raw_data)} items")
print(f"Global corpus size: {len(global_corpus)}")
print(f"Eval anchors with non-empty positives: {len(eval_items)}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")

Loaded 186 items
Global corpus size: 2229
Eval anchors with non-empty positives: 186
Checkpoint path: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/evaluation/../checkpoints/wikidata_model_semantic-20.pt


In [3]:
evaluate_sparqlce_checkpoint(
    checkpoint_path=CHECKPOINT_PATH,
    eval_items=eval_items,
    global_corpus=global_corpus,
    relevance_scores_per_item=relevance_scores_per_item,
    topks=TOPKS,
    batch_size=BATCH,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35427.27it/s]


Loading model from /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/evaluation/../checkpoints/wikidata_model_semantic-20.pt...

SPARQLCE
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1631  0.0371  0.0575  0.2043  0.3172    0.1631  0.1531
@5     0.1785  0.0714  0.0950  0.2212  0.3925    0.1785  0.1673
@10    0.1817  0.1383  0.1453  0.2425  0.5484    0.1817  0.1830
@20    0.1750  0.2678  0.1937  0.2492  0.6452    0.1750  0.2272
